# Trilha Classificação
## Análise de Métricas, Curva ROC, Comparação de Modelos e Síntese

**Projeto:** Home Credit Default Risk  
**Dataset:** `application_train.csv`  
**Target:** `TARGET` — 0 = pagou normalmente, 1 = dificuldade de pagamento  

## Setup - executar antes de qualquer etapa

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

df = pd.read_csv('../data/raw/application_train.csv')

thresh = len(df) * 0.5
df = df.dropna(thresh=thresh, axis=1)

for col in df.select_dtypes('object').columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

df = df.fillna(df.median(numeric_only=True))

X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Distribuição do target (total):\n{y.value_counts(normalize=True).round(4)}')

Train: (246008, 79) | Test: (61503, 79)
Distribuição do target (total):
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64


---
## Etapa 1 - Baseline e Métrica Principal

**Objetivo:** entender o desbalanceamento da base e definir a métrica que guiará todas as avaliações subsequentes.

Passos:
1. Calcular e visualizar a distribuição das classes (`value_counts(normalize=True)`)
2. Com base na proporção, justificar por que accuracy não é adequada
3. Definir a métrica principal do projeto e documentar a justificativa

In [ ]:
# 1. Distribuição das classes


In [ ]:
# 2. Visualização - gráfico de barras com proporções


**Métrica principal escolhida:**  
*(preencher aqui)*

**Justificativa:**  
*(preencher aqui)*

---
## Etapa 2 — Análise Profunda das Métricas

**Objetivo:** treinar um modelo baseline, examinar o classification report e a matriz de confusão, e interpretar os erros no contexto de negócio.

Passos:
1. Treinar um `LogisticRegression` simples (ou o modelo que o grupo preferir como baseline)
2. Exibir o `classification_report`
3. Plotar a Matriz de Confusão com `ConfusionMatrixDisplay`
4. Responder: o modelo erra mais como Falso Positivo ou Falso Negativo?
5. Responder: no contexto do projeto (crédito), qual tipo de erro é mais grave e por quê?

In [ ]:
# 1. Treinar modelo baseline


In [ ]:
# 2. Classification report


In [ ]:
# 3. Matriz de Confusão


**O modelo erra mais como:**  
*(Falso Positivo / Falso Negativo — preencher com análise dos números)*

**Erro mais grave no contexto do projeto:**  
*(preencher aqui — lembrar: FN = aprovamos um inadimplente, FP = negamos um bom pagador)*

---
## Etapa 3 — Curva ROC e AUC

**Objetivo:** avaliar o poder discriminativo do modelo baseline via probabilidades, independentemente do threshold de decisão.

Passos:
1. Calcular probabilidades com `predict_proba`
2. Plotar a Curva ROC com `RocCurveDisplay.from_predictions`
3. Calcular o AUC com `roc_auc_score`
4. Interpretar o AUC — o que ele significa para este problema?

In [6]:
# 1 + 2. Probabilidades e Curva ROC
y_probs = model_baseline.predict_proba(X_test)[:, 1]

RocCurveDisplay.from_predictions(y_test, y_probs)
plt.title('Curva ROC - Modelo Baseline')
plt.plot([0, 1], [0, 1], 'k--') # Linha de base
plt.show()

Calculando as probabilidades para os dados de teste...


NameError: name 'modelo' is not defined

>A AUC ou Area Under the Curve, resume em um único número a capacidade do modelo de distinguir as classes. Em resumo, é a area preenchida abaixo da curva roc.

In [ ]:
#AUC
from sklearn.metrics import roc_auc_score

# Calculando o valor da area abaixo do grafico.
auc_valor = roc_auc_score(y_test, y_probs)

print(f"O valor da AUC do seu modelo é: {auc_valor:.4f}")

**Interpretação do AUC:**  
*(preencher aqui — o que significa esse valor para o problema de aprovação de crédito?)*

---
## Etapa 4 — Comparação de Modelos

**Objetivo:** comparar ao menos 3 modelos via cross-validation e ajustar hiperparâmetros do melhor via GridSearchCV.

Passos:
1. Testar os 3 modelos com `cross_val_score(cv=5)` usando a métrica definida na Etapa 1
2. Montar tabela: Modelo | Média CV | Desvio Padrão
3. Identificar o melhor modelo
4. Ajustar o hiperparâmetro principal do melhor modelo com `GridSearchCV`

Modelos sugeridos: `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`

In [ ]:
# 1. Cross-validation para os 3 modelos
# dica: use scoring='roc_auc' ou a métrica definida na Etapa 1

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'DecisionTree':       DecisionTreeClassifier(random_state=RANDOM_STATE),
    'RandomForest':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

# iterar sobre models, calcular cross_val_score e guardar resultados


In [ ]:
# 2. Tabela de resultados: Modelo | Média CV | Desvio Padrão


In [ ]:
# 4. GridSearchCV no melhor modelo
# exemplo de param_grid para RandomForest:
# param_grid = {'max_depth': [5, 10, 20], 'min_samples_leaf': [1, 5, 10]}


---
## Etapa 5 — Síntese

**Objetivo:** consolidar os resultados, escolher o modelo final e documentar as decisões.

Passos:
1. Montar tabela final: Modelo | Métrica CV | Métrica Teste | AUC
2. Escolher o modelo para a próxima sprint e justificar em 2-3 linhas
3. Anotar o principal tipo de erro e o impacto real no problema

In [ ]:
# 1. Tabela final consolidada
# Modelo | Métrica CV | Métrica Teste | AUC


**Modelo escolhido para a próxima sprint:**  
*(nome do modelo)*

**Justificativa (2-3 linhas):**  
*(preencher aqui)*

**Principal tipo de erro identificado:**  
*(Falso Positivo / Falso Negativo — qual predomina e qual é o impacto no negócio)*